# HiMoDiT — EvaluationLoads the four trained stages, generates molecules, and measures:- **Validity, uniqueness, novelty** and their product- **Controllability** — how closely the generated molecules match the  requested property targets- **Structure** — heavy atoms, ring counts, molecular weight, as a  distribution-fidelity check alongside V·U·NRequires `01_train.ipynb` to have completed all four stages.

## 1. Environment

In [ ]:
!pip install -q rdkit torch tqdm matplotlib

In [ ]:
import os, systry:    from google.colab import drive    drive.mount('/content/drive', force_remount=False)    REPO = '/content/drive/MyDrive/himodit'    DATA = '/content/drive/MyDrive/himodit-data'except ImportError:    REPO = os.path.abspath('..')    DATA = os.path.join(REPO, 'data')if REPO not in sys.path:    sys.path.insert(0, REPO)CSV        = f'{DATA}/250k_rndm_zinc_drugs_clean_3.csv'LABELS     = f'{DATA}/labels.pkl'CKPT_ROOT  = f'{DATA}/checkpoints'PROPERTIES = ['logP', 'SAS']# Generation settings. N_SAMPLES >= 1000 for a reportable number:# smaller runs overstate uniqueness, since collisions grow with count.N_SAMPLES   = 1024BATCH_SIZE  = 64CFG_SCALE   = 1.5     # 1.0 disables guidance; higher sharpens conditioningTEMPERATURE = 1.0A1_STEPS    = 20A2_STEPS    = 20TERM_STEPS  = 8SEED        = 0# Clamp A3 branch parents to be causal. Off by default because the# reference numbers were produced without it. See docs/limitations.md.ENFORCE_CAUSAL_PARENT = Falsefor stage in ['a1', 'a3', 'a2', 'terminal']:    path = f'{CKPT_ROOT}/{stage}/best_model.pt'    mark = 'ok  ' if os.path.isfile(path) else 'MISSING'    size = f'{os.path.getsize(path)/1e6:.1f} MB' if os.path.isfile(path) else ''    print(f'  {mark} {stage:9s} {size}')

## 2. Load the modelLoads EMA weights where present — the EMA shadow generally samplesbetter than the raw weights.

In [ ]:
import torchfrom himodit.pipeline import HiMoDiTmodel = HiMoDiT.from_checkpoints(CKPT_ROOT, use_ema=True)print(f'\ndevice: {model.device}, condition dim: {model.condition_dim}')

## 3. Smoke testEight molecules with diagnostics. If this prints valid SMILES thecascade is wired correctly; if validity is near zero, the printedper-stage diagnostics localise which stage is failing.

In [ ]:
from rdkit import Chem, RDLoggerRDLogger.DisableLog('rdApp.*')torch.manual_seed(SEED)condition = torch.randn(8, model.condition_dim, device=model.device)smiles = model.generate_batch(    condition, cfg_scale=CFG_SCALE, temperature=TEMPERATURE,    a1_steps=A1_STEPS, a2_steps=A2_STEPS, term_steps=TERM_STEPS,    seed=SEED, enforce_causal_parent=ENFORCE_CAUSAL_PARENT, debug=True,)print('\ngenerated:')n_valid = 0for i, smi in enumerate(smiles):    if smi is None:        print(f'  [{i}] None (assembly failed)')        continue    mol = Chem.MolFromSmiles(smi)    if mol is None or mol.GetNumHeavyAtoms() == 0:        print(f'  [{i}] {smi!r}  (invalid)')        continue    print(f'  [{i}] {Chem.MolToSmiles(mol)}')    n_valid += 1print(f'\nvalid: {n_valid}/8')

## 4. GenerateConditions are drawn from N(0, 1) per axis, approximating the z-scoredtraining distribution.

In [ ]:
import timet0 = time.time()all_smiles, all_conditions = model.generate(    n=N_SAMPLES, batch_size=BATCH_SIZE, seed=SEED,    return_conditions=True,    cfg_scale=CFG_SCALE, temperature=TEMPERATURE,    a1_steps=A1_STEPS, a2_steps=A2_STEPS, term_steps=TERM_STEPS,    enforce_causal_parent=ENFORCE_CAUSAL_PARENT,)elapsed = time.time() - t0print(f'\n{len(all_smiles)} molecules in {elapsed:.0f}s '      f'({1000*elapsed/len(all_smiles):.0f} ms each)')

## 5. Validity, uniqueness, noveltyNovelty is measured against the training molecules, so the label file isloaded to recover them.Note the `empty (zero-atom)` line in the failure breakdown. RDKit parses`""` into a valid zero-atom molecule rather than returning `None`, sowithout an explicit check a failed scaffold decode would be scored as avalid sample. Anything other than zero there means decodes are failingupstream.

In [ ]:
import picklefrom himodit.metrics import (    compute_vun, format_vun, describe_molecules, training_set_from_labels,)with open(LABELS, 'rb') as f:    labels = pickle.load(f)train_canonical = training_set_from_labels(labels)print(f'{len(train_canonical):,} distinct training molecules\n')metrics = compute_vun(all_smiles, train_canonical=train_canonical)print(format_vun(metrics))structure = describe_molecules(all_smiles)if structure:    print('\nStructure of the valid samples')    for key, value in structure.items():        print(f'  {key:22s} {value:.2f}')

## 6. ControllabilityCorrelates the property target given to the model against the propertythe generated molecule actually has, both in z-scored units.The statistics come from the training CSV and must be the same ones usedduring preprocessing — different statistics measure correlation againsta shifted target and understate it.

In [ ]:
from himodit.metrics import (    compute_controllability, format_controllability, property_stats_from_csv,)stats = property_stats_from_csv(CSV, PROPERTIES)for prop, s in stats.items():    print(f"  {prop}: mean={s['mean']:.3f} std={s['std']:.3f}")control = compute_controllability(    all_smiles, all_conditions.numpy(), stats, property_axes=PROPERTIES,)print()print(format_controllability(control))

In [ ]:
import matplotlib.pyplot as pltaxes_with_data = [a for a, r in control.items() if 'targets' in r]fig, axs = plt.subplots(1, max(len(axes_with_data), 1),                        figsize=(5.5 * max(len(axes_with_data), 1), 5),                        squeeze=False)for ax, axis in zip(axs[0], axes_with_data):    res = control[axis]    ax.scatter(res['targets'], res['achieved'], alpha=0.3, s=10)    lim = max(abs(res['targets']).max(), abs(res['achieved']).max(), 3)    ax.plot([-lim, lim], [-lim, lim], 'k--', alpha=0.5, label='perfect steering')    ax.set_xlabel(f'requested {axis} (z-scored)')    ax.set_ylabel(f'achieved {axis} (z-scored)')    ax.set_title(f'{axis}:  r = {res["r"]:.3f}   (n={res["n"]})')    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)    ax.grid(alpha=0.3); ax.legend()plt.tight_layout(); plt.show()

## 7. Sample galleryAn eyeball check. Metrics can look healthy while the chemistry isstrange, so it is worth actually looking at what came out.

In [ ]:
from rdkit.Chem import Draw, AllChemgallery = sorted(    metrics['unique_smiles'],    key=lambda s: -Chem.MolFromSmiles(s).GetNumHeavyAtoms(),)[:20]mols = [Chem.MolFromSmiles(s) for s in gallery]for mol in mols:    AllChem.Compute2DCoords(mol)Draw.MolsToGridImage(    mols, molsPerRow=4, subImgSize=(280, 220),    legends=[f'{m.GetNumHeavyAtoms()} heavy atoms' for m in mols],)

## 8. SaveWrites the generated molecules with their targets, plus a metrics JSON.

In [ ]:
import jsonimport pandas as pdrows = []for i, smi in enumerate(all_smiles):    row = {'smiles': smi}    for j, prop in enumerate(PROPERTIES):        row[f'target_{prop}_z'] = float(all_conditions[i, j])    rows.append(row)pd.DataFrame(rows).to_csv(f'{DATA}/generated.csv', index=False)report = {k: v for k, v in metrics.items() if k != 'unique_smiles'}report['structure'] = structurereport['controllability'] = {    axis: {'r': res['r'], 'n': res['n']} for axis, res in control.items()}report['settings'] = {    'n': N_SAMPLES, 'cfg_scale': CFG_SCALE, 'temperature': TEMPERATURE,    'a1_steps': A1_STEPS, 'a2_steps': A2_STEPS, 'term_steps': TERM_STEPS,    'seed': SEED, 'enforce_causal_parent': ENFORCE_CAUSAL_PARENT,}with open(f'{DATA}/metrics.json', 'w') as f:    json.dump(report, f, indent=2)print(f'wrote {DATA}/generated.csv and {DATA}/metrics.json')

## Reading the results| Metric | Healthy | Worth investigating ||---|---|---|| Validity | ≥ 90% | < 70% || Uniqueness | ≥ 95% | < 90% — mode collapse || Novelty | ≥ 90% | < 80% — memorisation || logP r | ≥ 0.6 | < 0.3 — conditioning is not landing || SAS r | ≥ 0.4 | < 0.2 |**Low validity:** read the failure breakdown. Assembly failures point atthe layout stages producing scaffolds the decoder rejects; parse failurespoint at A2 assigning atom identities that break valence. A non-zero`empty` count means scaffold decodes are failing outright — try`ENFORCE_CAUSAL_PARENT = True`.**Low uniqueness** with high validity suggests guidance is too strong;try `CFG_SCALE = 1.0`.**Weak controllability** with high validity is the opposite problem —conditioning is being washed out. Try raising `CFG_SCALE`, and checkthat the property statistics here match the ones used at preprocessing.